# CS3807 – Deep Learning Laboratory — Experiment 5
## Comprehensive Study of CNN Training, Regularization, Optimization, Hyperparameter Tuning, Transfer Learning and Cross-Validation

This notebook runs every experiment required by the Experiment 5 lab manual and **exports all plots
(as PNG files) and all result tables (as CSV/JSON files)** into the `results/` folder.

**This version does NOT use the `tensorflow_datasets` package.** The Oxford-IIIT Pet dataset is
downloaded directly from its original source (the Visual Geometry Group at Oxford) as a plain
`.tar.gz` archive and parsed manually, avoiding the `tensorflow_datasets` import issues entirely.

**Workflow:**
1. Run this notebook top-to-bottom (GPU recommended; CPU will work but will be slow).
2. Everything is written to `./results/` — figures in `results/figures/`, tables in `results/tables/`.
3. Zip/download the `results/` folder and send it back so the final LaTeX report can be completed
   with the *actual* numbers and plots (no fabricated results will be used in the report).

**Notes on scale:** Default epoch counts / config grids are kept small (`QUICK_MODE = True`) so the
whole notebook can be run once end-to-end to sanity check it. Set `QUICK_MODE = False` in the config
cell to use the full grids from the manual (more epochs, all optimizers, all hyperparameter values,
full 5-fold CV). This will take substantially longer.


In [ ]:
# ============================================================
# 0. Environment check
# ============================================================
import subprocess, sys

def pip_install(pkg):
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", pkg], check=False)

try:
    import tensorflow as tf
except ImportError:
    pip_install("tensorflow")
    import tensorflow as tf

try:
    import sklearn
except ImportError:
    pip_install("scikit-learn")

import matplotlib
matplotlib.use("Agg")  # safe for headless export
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)
print("GPUs available:", tf.config.list_physical_devices('GPU'))


TensorFlow version: 2.20.0
GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# ============================================================
# 1. Global configuration
# ============================================================
import os, re, json, time, random, tarfile, urllib.request
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMG_SIZE = 224
CHANNELS = 3
NUM_CLASSES = 37          # Oxford-IIIT Pet: 37 breeds

RESULTS_DIR = "results"
FIG_DIR = os.path.join(RESULTS_DIR, "figures")
TBL_DIR = os.path.join(RESULTS_DIR, "tables")
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TBL_DIR, exist_ok=True)

# --- QUICK_MODE controls how big the experiment grid is ---
# True  -> fast sanity-check run (few epochs, subset of data, fewer configs)
# False -> full run following the manual's exact grids (slow)
QUICK_MODE = False

if QUICK_MODE:
    EPOCHS_SMALL   = 5     # used for init / regularization / BN / optimizer comparisons
    EPOCHS_TUNE    = 4     # used for hyperparameter sweeps
    EPOCHS_TRANSFER= 5     # feature extraction / fine-tuning
    EPOCHS_FINAL   = 6     # final retraining before test evaluation
    KFOLDS         = 5     # manual requires K=5; keep 5 but with fewer epochs per fold
    EPOCHS_CV      = 3
    TRAIN_SUBSET   = 800   # cap number of training images used (None = use all)
    BATCH_SIZE_DEFAULT = 32
else:
    EPOCHS_SMALL   = 10
    EPOCHS_TUNE    = 8
    EPOCHS_TRANSFER= 10
    EPOCHS_FINAL   = 10
    KFOLDS         = 5
    EPOCHS_CV      = 4
    TRAIN_SUBSET   = None
    BATCH_SIZE_DEFAULT = 32

print("QUICK_MODE =", QUICK_MODE)


QUICK_MODE = False


## 3. Dataset and Experimental Setup — Oxford-IIIT Pet Dataset

In [ ]:
# ============================================================
# Download the Oxford-IIIT Pet dataset directly from its original source
# (no tensorflow_datasets dependency). This downloads:
#   - images.tar.gz       (all pet images)
#   - annotations.tar.gz  (official trainval/test split + breed labels)
# and parses the annotation files ourselves.
# ============================================================
DATA_DIR = "oxford_pet_data"
os.makedirs(DATA_DIR, exist_ok=True)

IMAGES_URL = "https://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz"
ANNOTATIONS_URL = "https://www.robots.ox.ac.uk/~vgg/data/pets/data/annotations.tar.gz"

def download_and_extract(url, dest_dir):
    fname = os.path.join(dest_dir, os.path.basename(url))
    if not os.path.exists(fname):
        print("Downloading", url, "...")
        urllib.request.urlretrieve(url, fname)
        print("  done.")
    marker = fname + ".extracted"
    if not os.path.exists(marker):
        print("Extracting", fname, "...")
        with tarfile.open(fname) as tar:
            tar.extractall(dest_dir)
        open(marker, "w").close()
        print("  done.")

download_and_extract(IMAGES_URL, DATA_DIR)
download_and_extract(ANNOTATIONS_URL, DATA_DIR)

IMAGES_DIR = os.path.join(DATA_DIR, "images")
ANNOTATIONS_DIR = os.path.join(DATA_DIR, "annotations")

def parse_split_file(path):
    """Each non-comment line looks like:
    <image_id> <class_id 1-37> <species_id 1=cat,2=dog> <breed_id>
    """
    entries = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            image_id, class_id = parts[0], int(parts[1])
            entries.append((image_id, class_id - 1))  # zero-indexed class id
    return entries

trainval_entries = parse_split_file(os.path.join(ANNOTATIONS_DIR, "trainval.txt"))
test_entries_raw = parse_split_file(os.path.join(ANNOTATIONS_DIR, "test.txt"))

def breed_name_from_image_id(image_id):
    # image_id looks like 'Abyssinian_100' or 'yorkshire_terrier_12'
    return re.sub(r"_\d+$", "", image_id).replace("_", " ")

class_id_to_name = {}
for image_id, class_id in trainval_entries + test_entries_raw:
    if class_id not in class_id_to_name:
        class_id_to_name[class_id] = breed_name_from_image_id(image_id)
CLASS_NAMES = [class_id_to_name.get(i, f"class_{i}") for i in range(NUM_CLASSES)]

def to_paths(entries):
    out = []
    for image_id, class_id in entries:
        path = os.path.join(IMAGES_DIR, image_id + ".jpg")
        if os.path.exists(path):
            out.append((path, class_id))
    return out

trainval_entries = to_paths(trainval_entries)
test_entries = to_paths(test_entries_raw)

random.Random(SEED).shuffle(trainval_entries)
n_val = int(0.2 * len(trainval_entries))
val_entries = trainval_entries[:n_val]
train_entries = trainval_entries[n_val:]

if TRAIN_SUBSET is not None:
    train_entries = train_entries[:TRAIN_SUBSET]

print(f"train={len(train_entries)}  val={len(val_entries)}  test={len(test_entries)}")

def load_and_preprocess(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.keras.applications.mobilenet_v2.preprocess_input(image)
    label = tf.one_hot(label, NUM_CLASSES)
    return image, label

def make_pipeline_from_entries(entries, batch_size=BATCH_SIZE_DEFAULT, shuffle=False):
    paths = [e[0] for e in entries]
    labels = [e[1] for e in entries]
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(max(len(entries), 1), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_pipeline_from_entries(train_entries, shuffle=True)
val_ds   = make_pipeline_from_entries(val_entries)
test_ds  = make_pipeline_from_entries(test_entries)

dataset_summary = {
    "dataset": "Oxford-IIIT Pet Dataset",
    "source": IMAGES_URL,
    "num_classes": NUM_CLASSES,
    "image_size": f"{IMG_SIZE}x{IMG_SIZE}x{CHANNELS}",
    "n_train": len(train_entries),
    "n_val": len(val_entries),
    "n_test": len(test_entries),
    "preprocessing": "resize to 224x224, mobilenet_v2.preprocess_input normalization",
}
with open(os.path.join(TBL_DIR, "dataset_summary.json"), "w") as f:
    json.dump(dataset_summary, f, indent=2)

print(dataset_summary)


  done.
Extracting oxford_pet_data/images.tar.gz ...


/tmp/ipykernel_2672/3067259819.py:24: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(dest_dir)


  done.
  done.
Extracting oxford_pet_data/annotations.tar.gz ...
  done.
train=2944  val=736  test=3669
{'dataset': 'Oxford-IIIT Pet Dataset', 'source': 'https://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz', 'num_classes': 37, 'image_size': '224x224x3', 'n_train': 2944, 'n_val': 736, 'n_test': 3669, 'preprocessing': 'resize to 224x224, mobilenet_v2.preprocess_input normalization'}


## 5. Weight Initialization — Plot 1 (Training Loss) & Plot 2 (Validation Accuracy)

In [ ]:
# ============================================================
# Weight initialization: zero / random(normal) / xavier(glorot) / he
# Small classifier head only (base frozen), so that the initializer
# choice actually affects trainable weights in a controlled way.
# ============================================================
INIT_METHODS = {
    "Zero": tf.keras.initializers.Zeros(),
    "Random": tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.05, seed=SEED),
    "Xavier/Glorot": tf.keras.initializers.GlorotUniform(seed=SEED),
    "He": tf.keras.initializers.HeNormal(seed=SEED),
}

def build_model(initializer=None, dropout_rate=0.0, l2_reg=0.0,
                 use_batchnorm=False, trainable_base=False, optimizer="adam",
                 fine_tune_at=None):
    base = tf.keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, CHANNELS),
        include_top=False,
        weights="imagenet",
        pooling="avg",
    )
    base.trainable = trainable_base
    if trainable_base and fine_tune_at is not None:
        for layer in base.layers[:fine_tune_at]:
            layer.trainable = False

    reg = tf.keras.regularizers.l2(l2_reg) if l2_reg > 0 else None
    kwargs = {}
    if initializer is not None:
        kwargs["kernel_initializer"] = initializer

    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, CHANNELS))
    x = base(inputs, training=False if not trainable_base else None)
    if use_batchnorm:
        x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dense(128, activation="relu",
                               kernel_regularizer=reg, **kwargs)(x)
    if dropout_rate > 0:
        x = tf.keras.layers.Dropout(dropout_rate)(x)
    outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax",
                                     kernel_regularizer=reg, **kwargs)(x)
    model = tf.keras.Model(inputs, outputs)

    optimizers = {
        "sgd": tf.keras.optimizers.SGD(learning_rate=0.001),
        "momentum": tf.keras.optimizers.SGD(learning_rate=0.001, momentum=0.9),
        "rmsprop": tf.keras.optimizers.RMSprop(learning_rate=0.001),
        "adam": tf.keras.optimizers.Adam(learning_rate=0.001),
    }
    opt = optimizers[optimizer] if isinstance(optimizer, str) else optimizer

    model.compile(optimizer=opt, loss="categorical_crossentropy", metrics=["accuracy"])
    return model


class EpochLogger(tf.keras.callbacks.Callback):
    """Prints a clean one-line progress update after every epoch."""
    def __init__(self, tag):
        super().__init__()
        self.tag = tag

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        print(f"    [{self.tag}] epoch {epoch+1:>2}: "
              f"loss={logs.get('loss', float('nan')):.4f}  "
              f"acc={logs.get('accuracy', float('nan')):.4f}  "
              f"val_loss={logs.get('val_loss', float('nan')):.4f}  "
              f"val_acc={logs.get('val_accuracy', float('nan')):.4f}")


def run_and_time(model, train_ds, val_ds, epochs, tag="", verbose=0):
    t0 = time.time()
    hist = model.fit(
        train_ds, validation_data=val_ds, epochs=epochs, verbose=verbose,
        callbacks=[EpochLogger(tag)] if tag else [],
    )
    elapsed = time.time() - t0
    return hist, elapsed


print("=" * 70)
print("WEIGHT INITIALIZATION COMPARISON")
print(f"Training {len(INIT_METHODS)} models for {EPOCHS_SMALL} epochs each "
      f"({len(train_entries)} train / {len(val_entries)} val images)")
print("=" * 70)

init_histories = {}
init_times = {}
for i, (name, initializer) in enumerate(INIT_METHODS.items(), start=1):
    print(f"\n[{i}/{len(INIT_METHODS)}] Initializer: {name}")
    print("-" * 50)
    m = build_model(initializer=initializer, trainable_base=False, optimizer="adam")
    hist, elapsed = run_and_time(m, train_ds, val_ds, EPOCHS_SMALL, tag=name)
    init_histories[name] = hist.history
    init_times[name] = elapsed

    best_val_acc = max(hist.history["val_accuracy"])
    final_loss = hist.history["loss"][-1]
    print(f"  -> done in {elapsed:.1f}s | final train loss={final_loss:.4f} "
          f"| best val acc={best_val_acc*100:.2f}%")

# Save raw history
with open(os.path.join(TBL_DIR, "weight_init_history.json"), "w") as f:
    json.dump(init_histories, f, indent=2)

# ---- Formatted comparison summary ----
summary_rows = []
for name, h in init_histories.items():
    summary_rows.append({
        "Initializer": name,
        "Final Train Loss": h["loss"][-1],
        "Final Train Acc": h["accuracy"][-1],
        "Best Val Accuracy (%)": max(h["val_accuracy"]) * 100,
        "Final Val Loss": h["val_loss"][-1],
        "Time (s)": round(init_times[name], 1),
    })
init_summary_df = pd.DataFrame(summary_rows).sort_values(
    "Best Val Accuracy (%)", ascending=False
).reset_index(drop=True)
init_summary_df.to_csv(os.path.join(TBL_DIR, "weight_init_summary.csv"), index=False)

print("\n" + "=" * 70)
print("SUMMARY — ranked by best validation accuracy")
print("=" * 70)
print(init_summary_df.to_string(index=False))
print(f"\nBest performing initializer: {init_summary_df.iloc[0]['Initializer']}")

# Plot 1: Training Loss vs Epoch
plt.figure(figsize=(7,5))
for name, h in init_histories.items():
    plt.plot(range(1, len(h["loss"])+1), h["loss"], label=name, marker="o", markersize=3)
plt.xlabel("Epoch"); plt.ylabel("Training Loss")
plt.title("Training Loss vs. Epoch — Weight Initialization")
plt.legend(); plt.grid(alpha=0.3)
plt.savefig(os.path.join(FIG_DIR, "training_loss_initialization.png"), dpi=150, bbox_inches="tight")
plt.close()

# Plot 2: Validation Accuracy vs Epoch
plt.figure(figsize=(7,5))
for name, h in init_histories.items():
    plt.plot(range(1, len(h["val_accuracy"])+1), [a*100 for a in h["val_accuracy"]],
              label=name, marker="o", markersize=3)
plt.xlabel("Epoch"); plt.ylabel("Validation Accuracy (%)")
plt.title("Validation Accuracy vs. Epoch — Weight Initialization")
plt.legend(); plt.grid(alpha=0.3)
plt.savefig(os.path.join(FIG_DIR, "val_accuracy_initialization.png"), dpi=150, bbox_inches="tight")
plt.close()

print("\nSaved: training_loss_initialization.png, val_accuracy_initialization.png")
print("Saved: weight_init_summary.csv")

WEIGHT INITIALIZATION COMPARISON
Training 4 models for 10 epochs each (2944 train / 736 val images)

[1/4] Initializer: Zero
--------------------------------------------------
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
    [Zero] epoch  1: loss=3.6114  acc=0.0211  val_loss=3.6117  val_acc=0.0177
    [Zero] epoch  2: loss=3.6111  acc=0.0262  val_loss=3.6123  val_acc=0.0163
    [Zero] epoch  3: loss=3.6109  acc=0.0279  val_loss=3.6130  val_acc=0.0163
    [Zero] epoch  4: loss=3.6108  acc=0.0282  val_loss=3.6136  val_acc=0.0163
    [Zero] epoch  5: loss=3.6107  acc=0.0262  val_loss=3.6142  val_acc=0.0163
    [Zero] epoch  6: loss=3.6106  acc=0.0268  val_loss=3.6147  val_acc=0.0163
    [Zero] epoch  7: loss=3.6105  acc=0.0279  val_loss=3.6151  val_acc=0.0163
    [Zero] epoch  8: loss=3.6104  acc=0.0299  val_loss=3.6156  val_acc=0.0163
    [Zero] epoch  9: loss=3.6104  acc=0.0299  val_loss=3.6161  val_acc=0.0163
    [Zero] epoch 10: loss=3.6104  acc=0.0279  val_loss=3.6163  val_acc=0.

## 6. Regularization and Overfitting — Plot 3 (Accuracy) & Plot 4 (Loss)

In [ ]:
# ============================================================
# Regularization comparison: none / L2 / dropout / batchnorm
# ============================================================
REG_CONFIGS = {
    "No Regularization": dict(dropout_rate=0.0, l2_reg=0.0, use_batchnorm=False),
    "L2 Regularization": dict(dropout_rate=0.0, l2_reg=1e-3, use_batchnorm=False),
    "Dropout": dict(dropout_rate=0.5, l2_reg=0.0, use_batchnorm=False),
    "Batch Normalization": dict(dropout_rate=0.0, l2_reg=0.0, use_batchnorm=True),
}

reg_histories = {}
for name, cfg in REG_CONFIGS.items():
    print("Training with regularization:", name)
    m = build_model(trainable_base=False, optimizer="adam", **cfg)
    hist, _ = run_and_time(m, train_ds, val_ds, EPOCHS_SMALL)
    reg_histories[name] = hist.history

with open(os.path.join(TBL_DIR, "regularization_history.json"), "w") as f:
    json.dump(reg_histories, f, indent=2)

# Plot 3: Training & Validation Accuracy vs Epoch (one figure per config, overlaid)
fig, axes = plt.subplots(1, len(reg_histories), figsize=(5*len(reg_histories), 4), sharey=True)
if len(reg_histories) == 1:
    axes = [axes]
for ax, (name, h) in zip(axes, reg_histories.items()):
    ax.plot(h["accuracy"], label="Train Acc")
    ax.plot(h["val_accuracy"], label="Val Acc")
    ax.set_title(name); ax.set_xlabel("Epoch"); ax.grid(alpha=0.3)
axes[0].set_ylabel("Accuracy"); axes[0].legend()
plt.suptitle("Training and Validation Accuracy vs. Epoch — Regularization")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "accuracy_regularization.png"), dpi=150, bbox_inches="tight")
plt.close()

# Plot 4: Training & Validation Loss vs Epoch
fig, axes = plt.subplots(1, len(reg_histories), figsize=(5*len(reg_histories), 4), sharey=True)
if len(reg_histories) == 1:
    axes = [axes]
for ax, (name, h) in zip(axes, reg_histories.items()):
    ax.plot(h["loss"], label="Train Loss")
    ax.plot(h["val_loss"], label="Val Loss")
    ax.set_title(name); ax.set_xlabel("Epoch"); ax.grid(alpha=0.3)
axes[0].set_ylabel("Loss"); axes[0].legend()
plt.suptitle("Training and Validation Loss vs. Epoch — Regularization")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "loss_regularization.png"), dpi=150, bbox_inches="tight")
plt.close()

# Generalization gap table
gap_rows = []
for name, h in reg_histories.items():
    gap_rows.append({
        "Configuration": name,
        "Final Train Acc": h["accuracy"][-1],
        "Final Val Acc": h["val_accuracy"][-1],
        "Generalization Gap (Train-Val Acc)": h["accuracy"][-1] - h["val_accuracy"][-1],
        "Final Train Loss": h["loss"][-1],
        "Final Val Loss": h["val_loss"][-1],
    })
pd.DataFrame(gap_rows).to_csv(os.path.join(TBL_DIR, "regularization_generalization_gap.csv"), index=False)
print("Saved regularization plots and generalization-gap table.")


Training with regularization: No Regularization
Training with regularization: L2 Regularization
Training with regularization: Dropout
Training with regularization: Batch Normalization
Saved regularization plots and generalization-gap table.


## 7. Batch Normalization — Numerical Example & Plot 5

In [ ]:
# ============================================================
# Numerical example exactly as in the manual: x = [2,4,6,8]
# ============================================================
x = np.array([2, 4, 6, 8], dtype=float)
mu_B = x.mean()
sigma_B2 = ((x - mu_B) ** 2).mean()
eps = 1e-8
x_hat = (x - mu_B) / np.sqrt(sigma_B2 + eps)

gamma, beta = 1.0, 0.0
y = gamma * x_hat + beta

bn_numeric_example = {
    "x": x.tolist(),
    "mu_B": float(mu_B),
    "sigma_B2": float(sigma_B2),
    "x_hat": [float(v) for v in x_hat],
    "gamma": gamma,
    "beta": beta,
    "y": [float(v) for v in y],
}
with open(os.path.join(TBL_DIR, "batchnorm_numeric_example.json"), "w") as f:
    json.dump(bn_numeric_example, f, indent=2)
print(bn_numeric_example)

# ============================================================
# With BN vs Without BN — Plot 5 (Validation Accuracy vs Epoch)
# ============================================================
print("Training WITHOUT Batch Normalization...")
m_no_bn = build_model(trainable_base=False, use_batchnorm=False, optimizer="adam")
hist_no_bn, _ = run_and_time(m_no_bn, train_ds, val_ds, EPOCHS_SMALL)

print("Training WITH Batch Normalization...")
m_bn = build_model(trainable_base=False, use_batchnorm=True, optimizer="adam")
hist_bn, _ = run_and_time(m_bn, train_ds, val_ds, EPOCHS_SMALL)

bn_histories = {"Without BN": hist_no_bn.history, "With BN": hist_bn.history}
with open(os.path.join(TBL_DIR, "batchnorm_comparison_history.json"), "w") as f:
    json.dump(bn_histories, f, indent=2)

plt.figure(figsize=(7,5))
plt.plot([a*100 for a in hist_no_bn.history["val_accuracy"]], label="Without BN")
plt.plot([a*100 for a in hist_bn.history["val_accuracy"]], label="With BN")
plt.xlabel("Epoch"); plt.ylabel("Validation Accuracy (%)")
plt.title("With vs. Without Batch Normalization")
plt.legend(); plt.grid(alpha=0.3)
plt.savefig(os.path.join(FIG_DIR, "batchnorm_with_vs_without.png"), dpi=150, bbox_inches="tight")
plt.close()
print("Saved batchnorm_with_vs_without.png")


{'x': [2.0, 4.0, 6.0, 8.0], 'mu_B': 5.0, 'sigma_B2': 5.0, 'x_hat': [-1.341640785158233, -0.4472135950527444, 0.4472135950527444, 1.341640785158233], 'gamma': 1.0, 'beta': 0.0, 'y': [-1.341640785158233, -0.4472135950527444, 0.4472135950527444, 1.341640785158233]}
Training WITHOUT Batch Normalization...
Training WITH Batch Normalization...
Saved batchnorm_with_vs_without.png


## 8. Optimization Algorithms — Plot 6, Plot 7, and Optimizer Table

In [ ]:
# ============================================================
# SGD / Momentum / RMSProp / Adam comparison
# ============================================================
OPTIMIZERS = ["sgd", "momentum", "rmsprop", "adam"]
opt_histories = {}
opt_times = {}

for opt_name in OPTIMIZERS:
    print("Training with optimizer:", opt_name)
    m = build_model(trainable_base=False, optimizer=opt_name)
    hist, elapsed = run_and_time(m, train_ds, val_ds, EPOCHS_SMALL)
    opt_histories[opt_name] = hist.history
    opt_times[opt_name] = elapsed

with open(os.path.join(TBL_DIR, "optimizer_history.json"), "w") as f:
    json.dump(opt_histories, f, indent=2)

# Plot 6: Training Loss vs Epoch
plt.figure(figsize=(7,5))
for name, h in opt_histories.items():
    plt.plot(h["loss"], label=name)
plt.xlabel("Epoch"); plt.ylabel("Training Loss")
plt.title("Training Loss vs. Epoch — Optimizers")
plt.legend(); plt.grid(alpha=0.3)
plt.savefig(os.path.join(FIG_DIR, "training_loss_optimizers.png"), dpi=150, bbox_inches="tight")
plt.close()

# Plot 7: Validation Accuracy vs Epoch
plt.figure(figsize=(7,5))
for name, h in opt_histories.items():
    plt.plot([a*100 for a in h["val_accuracy"]], label=name)
plt.xlabel("Epoch"); plt.ylabel("Validation Accuracy (%)")
plt.title("Validation Accuracy vs. Epoch — Optimizers")
plt.legend(); plt.grid(alpha=0.3)
plt.savefig(os.path.join(FIG_DIR, "val_accuracy_optimizers.png"), dpi=150, bbox_inches="tight")
plt.close()

# Optimizer summary table
def epoch_to_converge(val_acc_list, tol=0.01):
    best = max(val_acc_list)
    for i, v in enumerate(val_acc_list):
        if v >= best - tol:
            return i + 1
    return len(val_acc_list)

rows = []
for name, h in opt_histories.items():
    rows.append({
        "Optimizer": name,
        "Final Loss": h["loss"][-1],
        "Best Val. Accuracy": max(h["val_accuracy"]),
        "Epoch to Converge": epoch_to_converge(h["val_accuracy"]),
        "Time (s)": round(opt_times[name], 2),
    })
opt_table = pd.DataFrame(rows)
opt_table.to_csv(os.path.join(TBL_DIR, "optimizer_comparison.csv"), index=False)
print(opt_table)


Training with optimizer: sgd
Training with optimizer: momentum
Training with optimizer: rmsprop
Training with optimizer: adam
  Optimizer  Final Loss  Best Val. Accuracy  Epoch to Converge  Time (s)
0       sgd    2.314035            0.510870                 10     99.82
1  momentum    0.246939            0.911685                  6    105.93
2   rmsprop    0.002874            0.914402                  5    108.77
3      adam    0.007023            0.915761                  4    102.73


## 9. CNN Hyperparameter Tuning — Plot 8, Plot 9, Plot 10 (one-variable-at-a-time)

In [ ]:
# ============================================================
# One-variable-at-a-time hyperparameter sweeps.
# Baseline fixed settings: lr=0.001, batch_size=32, dropout=0.0, optimizer=adam
# ============================================================
LR_VALUES = [0.001, 0.0001]
BATCH_SIZES = [16, 32, 64]
DROPOUT_RATES = [0.0, 0.25, 0.5]

def train_with_lr(lr, epochs=EPOCHS_TUNE):
    m = build_model(trainable_base=False,
                     optimizer=tf.keras.optimizers.Adam(learning_rate=lr))
    hist, _ = run_and_time(m, train_ds, val_ds, epochs)
    return max(hist.history["val_accuracy"])

def train_with_batch_size(bs, epochs=EPOCHS_TUNE):
    tds = make_pipeline_from_entries(train_entries, batch_size=bs, shuffle=True)
    vds = make_pipeline_from_entries(val_entries, batch_size=bs)
    m = build_model(trainable_base=False, optimizer="adam")
    hist, _ = run_and_time(m, tds, vds, epochs)
    return max(hist.history["val_accuracy"])

def train_with_dropout(dr, epochs=EPOCHS_TUNE):
    m = build_model(trainable_base=False, dropout_rate=dr, optimizer="adam")
    hist, _ = run_and_time(m, train_ds, val_ds, epochs)
    return max(hist.history["val_accuracy"])

lr_results = {lr: train_with_lr(lr) for lr in LR_VALUES}
bs_results = {bs: train_with_batch_size(bs) for bs in BATCH_SIZES}
do_results = {dr: train_with_dropout(dr) for dr in DROPOUT_RATES}

tuning_summary = {
    "learning_rate": {str(k): float(v) for k, v in lr_results.items()},
    "batch_size": {str(k): float(v) for k, v in bs_results.items()},
    "dropout_rate": {str(k): float(v) for k, v in do_results.items()},
}
with open(os.path.join(TBL_DIR, "hyperparameter_tuning_results.json"), "w") as f:
    json.dump(tuning_summary, f, indent=2)

# Plot 8: Learning Rate vs Validation Accuracy
plt.figure(figsize=(6,4))
plt.plot([str(k) for k in lr_results.keys()], [v*100 for v in lr_results.values()], marker="o")
plt.xlabel("Learning Rate"); plt.ylabel("Validation Accuracy (%)")
plt.title("Learning Rate vs. Validation Accuracy")
plt.grid(alpha=0.3)
plt.savefig(os.path.join(FIG_DIR, "lr_vs_val_accuracy.png"), dpi=150, bbox_inches="tight")
plt.close()

# Plot 9: Batch Size vs Validation Accuracy
plt.figure(figsize=(6,4))
plt.plot([str(k) for k in bs_results.keys()], [v*100 for v in bs_results.values()], marker="o")
plt.xlabel("Batch Size"); plt.ylabel("Validation Accuracy (%)")
plt.title("Batch Size vs. Validation Accuracy")
plt.grid(alpha=0.3)
plt.savefig(os.path.join(FIG_DIR, "batchsize_vs_val_accuracy.png"), dpi=150, bbox_inches="tight")
plt.close()

# Plot 10: Dropout Rate vs Validation Accuracy
plt.figure(figsize=(6,4))
plt.plot([str(k) for k in do_results.keys()], [v*100 for v in do_results.values()], marker="o")
plt.xlabel("Dropout Rate"); plt.ylabel("Validation Accuracy (%)")
plt.title("Dropout Rate vs. Validation Accuracy")
plt.grid(alpha=0.3)
plt.savefig(os.path.join(FIG_DIR, "dropout_vs_val_accuracy.png"), dpi=150, bbox_inches="tight")
plt.close()

print(tuning_summary)


{'learning_rate': {'0.001': 0.9130434989929199, '0.0001': 0.9103260636329651}, 'batch_size': {'16': 0.91847825050354, '32': 0.9239130616188049, '64': 0.9130434989929199}, 'dropout_rate': {'0.0': 0.9089673757553101, '0.25': 0.9225543737411499, '0.5': 0.917119562625885}}


## 10. Transfer Learning and Fine-Tuning — Plot 11, Plot 12

In [ ]:
# ============================================================
# Case A: Feature Extraction (frozen base)
# Case B: Fine-Tuning (unfreeze upper layers, smaller LR)
# ============================================================
print("Case A: Feature Extraction")
m_fe = build_model(trainable_base=False, optimizer="adam")
hist_fe, _ = run_and_time(m_fe, train_ds, val_ds, EPOCHS_TRANSFER)

print("Case B: Fine-Tuning")
# Determine a fine-tuning cut point: unfreeze roughly the last third of base layers
base_probe = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, CHANNELS), include_top=False, weights=None)
n_base_layers = len(base_probe.layers)
fine_tune_at = int(n_base_layers * 0.66)

m_ft = build_model(trainable_base=True, fine_tune_at=fine_tune_at,
                    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4))
hist_ft, _ = run_and_time(m_ft, train_ds, val_ds, EPOCHS_TRANSFER)

transfer_histories = {"Feature Extraction": hist_fe.history, "Fine-Tuning": hist_ft.history}
with open(os.path.join(TBL_DIR, "transfer_learning_history.json"), "w") as f:
    json.dump(transfer_histories, f, indent=2)

# Plot 11: Feature Extraction vs Fine-Tuning (Validation Accuracy)
plt.figure(figsize=(7,5))
plt.plot([a*100 for a in hist_fe.history["val_accuracy"]], label="Feature Extraction")
plt.plot([a*100 for a in hist_ft.history["val_accuracy"]], label="Fine-Tuning")
plt.xlabel("Epoch"); plt.ylabel("Validation Accuracy (%)")
plt.title("Feature Extraction vs. Fine-Tuning")
plt.legend(); plt.grid(alpha=0.3)
plt.savefig(os.path.join(FIG_DIR, "feature_extraction_vs_finetuning.png"), dpi=150, bbox_inches="tight")
plt.close()

# Plot 12: Training and Validation Loss (Fine-Tuning stage)
plt.figure(figsize=(7,5))
plt.plot(hist_ft.history["loss"], label="Train Loss (Fine-Tuning)")
plt.plot(hist_ft.history["val_loss"], label="Val Loss (Fine-Tuning)")
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.title("Training and Validation Loss — Fine-Tuning")
plt.legend(); plt.grid(alpha=0.3)
plt.savefig(os.path.join(FIG_DIR, "loss_finetuning.png"), dpi=150, bbox_inches="tight")
plt.close()

print("Feature Extraction best val acc:", max(hist_fe.history["val_accuracy"]))
print("Fine-Tuning best val acc:", max(hist_ft.history["val_accuracy"]))


Case A: Feature Extraction
Case B: Fine-Tuning
Feature Extraction best val acc: 0.917119562625885
Fine-Tuning best val acc: 0.89673912525177


## 11. K-Fold Cross-Validation — Table & Plot 13

In [ ]:
# ============================================================
# 5-fold CV on a small set of candidate configurations selected from
# the preceding studies. Edit CANDIDATE_CONFIGS to match whichever
# configurations were actually found promising above.
#
# The CV pool is the full trainval set (train_entries + val_entries),
# i.e. everything except the held-out test set. Folds are built from
# plain (path, label) lists and streamed through tf.data, so nothing
# is loaded fully into memory at once.
# ============================================================
from sklearn.model_selection import KFold

CANDIDATE_CONFIGS = {
    "C1_baseline_adam":        dict(optimizer="adam", dropout_rate=0.0, l2_reg=0.0, use_batchnorm=False),
    "C2_dropout_adam":         dict(optimizer="adam", dropout_rate=0.5, l2_reg=0.0, use_batchnorm=False),
    "C3_bn_adam":              dict(optimizer="adam", dropout_rate=0.0, l2_reg=0.0, use_batchnorm=True),
    "C4_l2_rmsprop":           dict(optimizer="rmsprop", dropout_rate=0.0, l2_reg=1e-3, use_batchnorm=False),
}

cv_pool = train_entries + val_entries
indices = np.arange(len(cv_pool))
kf = KFold(n_splits=KFOLDS, shuffle=True, random_state=SEED)

def ds_from_indices(idx_list, batch_size=BATCH_SIZE_DEFAULT, shuffle=False):
    subset = [cv_pool[i] for i in idx_list]
    return make_pipeline_from_entries(subset, batch_size=batch_size, shuffle=shuffle)

cv_results = {}
for cfg_name, cfg in CANDIDATE_CONFIGS.items():
    fold_accs = []
    print(f"=== 5-fold CV for {cfg_name} ===")
    for fold_i, (train_idx, val_idx) in enumerate(kf.split(indices), start=1):
        tds = ds_from_indices(train_idx, shuffle=True)
        vds = ds_from_indices(val_idx)
        m = build_model(trainable_base=False, **cfg)
        hist, _ = run_and_time(m, tds, vds, EPOCHS_CV, verbose=0)
        best_val_acc = max(hist.history["val_accuracy"])
        fold_accs.append(best_val_acc)
        print(f"  Fold {fold_i}: val_acc={best_val_acc:.4f}")
    cv_results[cfg_name] = fold_accs

cv_rows = []
for cfg_name, accs in cv_results.items():
    mean_acc = float(np.mean(accs))
    sd_acc = float(np.std(accs))
    row = {"Configuration": cfg_name}
    for i, a in enumerate(accs, start=1):
        row[f"F{i}"] = a
    row["Mean"] = mean_acc
    row["SD"] = sd_acc
    row["Mean ± SD"] = f"{mean_acc:.4f} ± {sd_acc:.4f}"
    cv_rows.append(row)

cv_table = pd.DataFrame(cv_rows)
cv_table.to_csv(os.path.join(TBL_DIR, "kfold_cv_results.csv"), index=False)
print(cv_table)

# Plot 13: 5-Fold CV accuracy with SD error bars
plt.figure(figsize=(7,5))
names = list(cv_results.keys())
means = [np.mean(cv_results[n]) * 100 for n in names]
sds = [np.std(cv_results[n]) * 100 for n in names]
plt.bar(names, means, yerr=sds, capsize=5)
plt.xlabel("Hyperparameter Configuration"); plt.ylabel("Mean Validation Accuracy (%)")
plt.title("5-Fold Cross-Validation Accuracy")
plt.xticks(rotation=20)
plt.grid(axis="y", alpha=0.3)
plt.savefig(os.path.join(FIG_DIR, "kfold_cv_accuracy.png"), dpi=150, bbox_inches="tight")
plt.close()


=== 5-fold CV for C1_baseline_adam ===
  Fold 1: val_acc=0.9253
  Fold 2: val_acc=0.9062
  Fold 3: val_acc=0.9103
  Fold 4: val_acc=0.9035
  Fold 5: val_acc=0.9076
=== 5-fold CV for C2_dropout_adam ===
  Fold 1: val_acc=0.9171
  Fold 2: val_acc=0.9049
  Fold 3: val_acc=0.9185
  Fold 4: val_acc=0.8927
  Fold 5: val_acc=0.9130
=== 5-fold CV for C3_bn_adam ===
  Fold 1: val_acc=0.9062
  Fold 2: val_acc=0.9090
  Fold 3: val_acc=0.9130
  Fold 4: val_acc=0.9076
  Fold 5: val_acc=0.9158
=== 5-fold CV for C4_l2_rmsprop ===
  Fold 1: val_acc=0.9076
  Fold 2: val_acc=0.9022
  Fold 3: val_acc=0.9198
  Fold 4: val_acc=0.8954
  Fold 5: val_acc=0.8967
      Configuration        F1        F2        F3        F4        F5  \
0  C1_baseline_adam  0.925272  0.906250  0.910326  0.903533  0.907609   
1   C2_dropout_adam  0.917120  0.904891  0.918478  0.892663  0.913043   
2        C3_bn_adam  0.906250  0.908967  0.913043  0.907609  0.915761   
3     C4_l2_rmsprop  0.907609  0.902174  0.919837  0.895380  0

## 12. Final Model Evaluation — Table & Plot 14 (Confusion Matrix)

In [ ]:
# ============================================================
# Select the configuration with the best mean CV accuracy, retrain on
# the FULL training data, then evaluate ONCE on the untouched test set.
# ============================================================
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, ConfusionMatrixDisplay)

best_cfg_name = max(cv_results, key=lambda k: np.mean(cv_results[k]))
best_cfg = CANDIDATE_CONFIGS[best_cfg_name]
print("Best configuration selected by CV:", best_cfg_name, best_cfg)

final_model = build_model(trainable_base=False, **best_cfg)
t0 = time.time()
final_hist = final_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FINAL, verbose=0)
training_time = time.time() - t0

# Evaluate on the untouched test set
y_true, y_pred = [], []
for images, labels in test_ds:
    preds = final_model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

test_accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average="macro", zero_division=0)
recall = recall_score(y_true, y_pred, average="macro", zero_division=0)
f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
num_params = final_model.count_params()

mean_cv_acc = float(np.mean(cv_results[best_cfg_name]))
cv_sd = float(np.std(cv_results[best_cfg_name]))

final_eval = {
    "Selected Configuration": best_cfg_name,
    "Mean CV Accuracy": mean_cv_acc,
    "CV Standard Deviation": cv_sd,
    "Test Accuracy": float(test_accuracy),
    "Precision": float(precision),
    "Recall": float(recall),
    "F1-score": float(f1),
    "Training Time (s)": round(training_time, 2),
    "Number of Parameters": int(num_params),
}
pd.DataFrame([final_eval]).to_csv(os.path.join(TBL_DIR, "final_model_evaluation.csv"), index=False)
print(final_eval)

# Plot 14: Confusion Matrix
cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
fig, ax = plt.subplots(figsize=(14, 12))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot(ax=ax, xticks_rotation=90, colorbar=True, cmap="Blues")
plt.title("Confusion Matrix — Final Model on Test Set")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "confusion_matrix.png"), dpi=150, bbox_inches="tight")
plt.close()

np.savetxt(os.path.join(TBL_DIR, "confusion_matrix_raw.csv"), cm, delimiter=",", fmt="%d")
print("Saved confusion_matrix.png and confusion_matrix_raw.csv")


Best configuration selected by CV: C1_baseline_adam {'optimizer': 'adam', 'dropout_rate': 0.0, 'l2_reg': 0.0, 'use_batchnorm': False}
{'Selected Configuration': 'C1_baseline_adam', 'Mean CV Accuracy': 0.910597825050354, 'CV Standard Deviation': 0.007657068617272017, 'Test Accuracy': 0.8947942218588171, 'Precision': 0.895490108116664, 'Recall': 0.8940362741123941, 'F1-score': 0.8935582504315631, 'Training Time (s)': 105.49, 'Number of Parameters': 2426725}
Saved confusion_matrix.png and confusion_matrix_raw.csv


## 13. Overall Results Table

In [ ]:
# ============================================================
# Compile the overall results table across all experiment stages.
# ============================================================
def safe_best_val_acc(hist_dict):
    return max(hist_dict["val_accuracy"]) if hist_dict and "val_accuracy" in hist_dict else None

overall_rows = [
    {
        "Configuration": "Baseline",
        "CV Accuracy": None, "SD": None,
        "Test Accuracy": None, "Training Time": None,
    },
    {
        "Configuration": "Best Initialization",
        "CV Accuracy": None, "SD": None,
        "Test Accuracy": max([safe_best_val_acc(h) for h in init_histories.values()]),
        "Training Time": None,
    },
    {
        "Configuration": "Best Regularization",
        "CV Accuracy": None, "SD": None,
        "Test Accuracy": max([safe_best_val_acc(h) for h in reg_histories.values()]),
        "Training Time": None,
    },
    {
        "Configuration": "Best Optimizer",
        "CV Accuracy": None, "SD": None,
        "Test Accuracy": max([safe_best_val_acc(h) for h in opt_histories.values()]),
        "Training Time": None,
    },
    {
        "Configuration": "Best Hyperparameters",
        "CV Accuracy": None, "SD": None,
        "Test Accuracy": max(list(lr_results.values()) + list(bs_results.values()) + list(do_results.values())),
        "Training Time": None,
    },
    {
        "Configuration": "Fine-Tuned Model",
        "CV Accuracy": None, "SD": None,
        "Test Accuracy": safe_best_val_acc(hist_ft.history),
        "Training Time": None,
    },
    {
        "Configuration": f"Final Selected ({best_cfg_name})",
        "CV Accuracy": mean_cv_acc, "SD": cv_sd,
        "Test Accuracy": test_accuracy,
        "Training Time": round(training_time, 2),
    },
]
overall_table = pd.DataFrame(overall_rows)
overall_table.to_csv(os.path.join(TBL_DIR, "overall_results.csv"), index=False)
print(overall_table)


                       Configuration  CV Accuracy        SD  Test Accuracy  \
0                           Baseline          NaN       NaN            NaN   
1                Best Initialization          NaN       NaN       0.915761   
2                Best Regularization          NaN       NaN       0.913043   
3                     Best Optimizer          NaN       NaN       0.915761   
4               Best Hyperparameters          NaN       NaN       0.923913   
5                   Fine-Tuned Model          NaN       NaN       0.896739   
6  Final Selected (C1_baseline_adam)     0.910598  0.007657       0.894794   

   Training Time  
0            NaN  
1            NaN  
2            NaN  
3            NaN  
4            NaN  
5            NaN  
6         105.49  


## 16. Additional Exercise — Two New Configurations, 5-Fold CV

In [ ]:
# ============================================================
# Two NEW combinations of learning rate / dropout / batch size /
# fine-tuning strategy, compared with the previously selected
# configuration using 5-fold cross-validation.
# ============================================================
EXTRA_CONFIGS = {
    "Extra1_lowLR_highDropout": dict(
        dropout_rate=0.5,
        learning_rate=0.0001,          # <-- store the LR, not an optimizer instance
        trainable_base=False,
        fine_tune_at=None,
    ),
    "Extra2_finetune_smallLR": dict(
        dropout_rate=0.25,
        learning_rate=1e-5,
        trainable_base=True,
        fine_tune_at=fine_tune_at,
    ),
}

extra_cv_results = {}
for cfg_name, cfg in EXTRA_CONFIGS.items():
    fold_accs = []
    print(f"=== 5-fold CV for {cfg_name} ===")
    for fold_i, (train_idx, val_idx) in enumerate(kf.split(indices), start=1):
        tds = ds_from_indices(train_idx, shuffle=True)
        vds = ds_from_indices(val_idx)

        # Build a brand-new optimizer for THIS fold's model — never reuse one
        fresh_optimizer = tf.keras.optimizers.Adam(learning_rate=cfg["learning_rate"])

        m = build_model(
            trainable_base=cfg["trainable_base"],
            fine_tune_at=cfg["fine_tune_at"],
            dropout_rate=cfg["dropout_rate"],
            optimizer=fresh_optimizer,
        )
        hist, _ = run_and_time(m, tds, vds, EPOCHS_CV, verbose=0)
        best_val_acc = max(hist.history["val_accuracy"])
        fold_accs.append(best_val_acc)
        print(f"  Fold {fold_i}: val_acc={best_val_acc:.4f}")
    extra_cv_results[cfg_name] = fold_accs

extra_rows = []
for cfg_name, accs in extra_cv_results.items():
    mean_acc, sd_acc = float(np.mean(accs)), float(np.std(accs))
    row = {"Configuration": cfg_name}
    for i, a in enumerate(accs, start=1):
        row[f"F{i}"] = a
    row["Mean"] = mean_acc
    row["SD"] = sd_acc
    extra_rows.append(row)

# Include the previously selected configuration for direct comparison
extra_rows.append({
    "Configuration": f"Previously Selected ({best_cfg_name})",
    **{f"F{i}": a for i, a in enumerate(cv_results[best_cfg_name], start=1)},
    "Mean": mean_cv_acc, "SD": cv_sd,
})

extra_table = pd.DataFrame(extra_rows)
extra_table.to_csv(os.path.join(TBL_DIR, "additional_exercise_results.csv"), index=False)
print(extra_table)

=== 5-fold CV for Extra1_lowLR_highDropout ===
  Fold 1: val_acc=0.8370
  Fold 2: val_acc=0.8519


## Export Summary

In [ ]:
# ============================================================
# List every exported artifact so it is easy to confirm what to send back.
# ============================================================
import os, shutil

# Re-define these variables to ensure they are available within this cell's scope
RESULTS_DIR = "results"
FIG_DIR = os.path.join(RESULTS_DIR, "figures")
TBL_DIR = os.path.join(RESULTS_DIR, "tables")

print("Figures exported to:", FIG_DIR)
for f in sorted(os.listdir(FIG_DIR)):
    print(" -", f)

print()
print("Tables exported to:", TBL_DIR)
for f in sorted(os.listdir(TBL_DIR)):
    print(" -", f)

# Optional: zip everything up for convenient download
shutil.make_archive("experiment5_results", "zip", RESULTS_DIR)
print()
print("Zipped archive created: experiment5_results.zip")

Figures exported to: results/figures
 - accuracy_regularization.png
 - batchnorm_with_vs_without.png
 - batchsize_vs_val_accuracy.png
 - confusion_matrix.png
 - dropout_vs_val_accuracy.png
 - feature_extraction_vs_finetuning.png
 - kfold_cv_accuracy.png
 - loss_finetuning.png
 - loss_regularization.png
 - lr_vs_val_accuracy.png
 - training_loss_initialization.png
 - training_loss_optimizers.png
 - val_accuracy_initialization.png
 - val_accuracy_optimizers.png

Tables exported to: results/tables
 - batchnorm_comparison_history.json
 - batchnorm_numeric_example.json
 - confusion_matrix_raw.csv
 - dataset_summary.json
 - final_model_evaluation.csv
 - hyperparameter_tuning_results.json
 - kfold_cv_results.csv
 - optimizer_comparison.csv
 - optimizer_history.json
 - overall_results.csv
 - regularization_generalization_gap.csv
 - regularization_history.json
 - transfer_learning_history.json
 - weight_init_history.json
 - weight_init_summary.csv

Zipped archive created: experiment5_results.zi

In [ ]:
print(1)